# Install dependencies

In [1]:
!pip install beautifulsoup4

In [2]:
!pip install selenium

In [3]:
!pip install pandas

In [4]:
!pip install pyarrow

In [5]:
!pip install Pillow

In [6]:
!pip install requests

In [7]:
# Cell 1: Import necessary libraries
import os
import time
import csv
import requests
from bs4 import BeautifulSoup

In [8]:
# Cell 2: Function to fetch all links from a given URL
def get_all_links(url):
    """Fetch all links from the given URL."""
    try:
        response = requests.get(url)
        response.raise_for_status()  # Ensure the request was successful
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract all <a> tags with href attributes
        links = [a['href'] for a in soup.find_all('a', href=True)]
        return links
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []


In [9]:
# Cell 3: Function to save links to a CSV file
def save_links_to_csv(links, csv_file_path):
    """Save a list of links to a CSV file."""
    try:
        with open(csv_file_path, mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["link"])  # Header
            writer.writerows([[link] for link in links])
        print(f"Links saved to {csv_file_path}")
    except Exception as e:
        print(f"Error saving links to CSV: {e}")


In [10]:
# Cell 4: Function to download images from a URL (with delay)
def download_image(image_url, folder, index):
    """Download an image and save it to the specified folder."""
    try:
        # Wait 3 seconds before downloading each image
        time.sleep(1)

        # Create folder if it doesn't exist
        if not os.path.exists(folder):
            os.makedirs(folder)

        file_name = os.path.join(folder, f"image_{index + 1}.jpg")
        response = requests.get(image_url, stream=True)
        if response.status_code == 200:
            with open(file_name, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            print(f"Image saved to: {file_name}")
        else:
            print(f"Error downloading image: {image_url} (Status code: {response.status_code})")
    except Exception as e:
        print(f"Error downloading image: {image_url} ({e})")


In [11]:
# Cell 5: Function to process a URL, download images from it
def process_url(url, folder):
    """Fetch and download images from a URL."""
    try:
        htmldata = requests.get(url).text
        soup = BeautifulSoup(htmldata, 'html.parser')

        for index, img_tag in enumerate(soup.find_all('img')):
            image_url = img_tag.get('src')
            if image_url:
                download_image(image_url, folder, index)
    except Exception as e:
        print(f"Error processing URL: {url} ({e})")


In [12]:
# Cell 6: Main code to fetch links, save to CSV, and download images
def main():
    # URL of the main page
    main_url = "https://scryfall.com/sets"

    # Get all links from the main page
    print(f"Fetching links from {main_url}...")
    links = get_all_links(main_url)

    # Convert relative links to absolute links
    absolute_links = [requests.compat.urljoin(main_url, link) for link in links]

    # Save the links to a CSV file
    csv_file_path = "scryfall_links.DbzFusionWorldcsv"
    save_links_to_csv(absolute_links, csv_file_path)

    # Base folder to save images
    base_folder = "downloaded_images"

    # Process each link to download images into separate folders
    for link in absolute_links:
        folder_name = os.path.join(base_folder, link.split("/")[-1])  # Use the last part of the URL as folder name
        print(f"Processing images from: {link}")
        process_url(link, folder_name)
        print("Waiting for 3 seconds before processing the next URL...")
        time.sleep(3)



In [13]:
# Call main function
main()

Fetching links from https://scryfall.com/sets...
Links saved to scryfall_links.csv
Processing images from: https://scryfall.com/sets#main
Waiting for 3 seconds before processing the next URL...
Processing images from: https://scryfall.com/sets#footer
Waiting for 3 seconds before processing the next URL...
Processing images from: https://scryfall.com/
Image saved to: downloaded_images\image_1.jpg
Image saved to: downloaded_images\image_2.jpg
Image saved to: downloaded_images\image_3.jpg
Image saved to: downloaded_images\image_4.jpg
Image saved to: downloaded_images\image_5.jpg
Image saved to: downloaded_images\image_6.jpg
Image saved to: downloaded_images\image_7.jpg
Waiting for 3 seconds before processing the next URL...
Processing images from: https://scryfall.com/advanced
Waiting for 3 seconds before processing the next URL...
Processing images from: https://scryfall.com/docs/syntax
Waiting for 3 seconds before processing the next URL...
Processing images from: https://scryfall.com/s


KeyboardInterrupt

